# Genotype–Phenotype Heterogeneity and Heterogeneity-Driven Infertility Management in Non-Classical 21-Hydroxylase Deficiency: A Clinical Analysis and Literature Review Exploration with `mlcroissant`
This notebook guides you through loading and exploring the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
Dataset defined via Croissant schema JSON-LD:
https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json

In [ ]:
# Ensure mlcroissant library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json'

# Load the dataset package metadata
dataset = mlc.Dataset(croissant_url)
# Access the metadata object (not subscripting)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review all available record sets, fields, and their `@id` values.

**Note:** Each record set, field, and column is referenced by its `@id`.

In [ ]:
# View available record sets, fields, and their IDs
record_sets = []
for rs in dataset.record_sets():
    print(f"RecordSet @id: {rs['@id']}")
    record_sets.append(rs['@id'])
    print("  Fields:")
    for field in rs.get('field', []):
        # If field is a dict
        if isinstance(field, dict) and '@id' in field:
            print(f"    Field @id: {field['@id']} (name: {field.get('name', '')})")
        else:
            print(f"    Field @id: {field}")
    print()
# Show available columns for each record set
for rs in dataset.record_sets():
    print(f"RecordSet @id: {rs['@id']}")
    columns = rs.get('column', [])
    if columns:
        print("  Columns:")
        for col in columns:
            if isinstance(col, dict) and '@id' in col:
                print(f"    Column @id: {col['@id']} (name: {col.get('name', '')})")
            else:
                print(f"    Column @id: {col}")
    else:
        print("  No columns listed.")
    print()

## 3. Data Extraction
Load data from each record set into a pandas DataFrame for analysis.

Refer to record set and field `@id`s from above overview. Example below shows how to extract for all record sets.

In [ ]:
# Create a dictionary to hold DataFrames for each record set
dataframes = {}

# If record_sets list is empty, populate from overview above
if not record_sets:
    record_sets = [rs['@id'] for rs in dataset.record_sets()]

for record_set_id in record_sets:
    records = list(dataset.records(record_set=record_set_id))
    if records:
        dataframes[record_set_id] = pd.DataFrame(records)
        print(f"Columns in {record_set_id}: {dataframes[record_set_id].columns.tolist()}")
        print(f"Preview of {record_set_id} records:")
        print(dataframes[record_set_id].head())
    else:
        print(f"No records found for record set {record_set_id}.")

## 4. Exploratory Data Analysis (EDA)
Apply common processing steps: filtering by criteria, normalizing numeric fields, grouping by key attributes.

Below, we select a numeric field (e.g., age at diagnosis) by its column `@id` from the first record set (Table 1).

In [ ]:
# Example: Work with first record set containing clinical features
if len(record_sets) > 0:
    rs_id = record_sets[0]  # Table 1
    df = dataframes.get(rs_id, pd.DataFrame())

    # List columns with their @id
    print(f"Columns available in {rs_id}:")
    for col in df.columns:
        print(col)

    # Suppose age_at_diagnosis is '@id:table1/column/age_at_diagnosis', adjust as seen above
    # Try to identify proper numeric column
    numeric_field = None
    for col in df.columns:
        if 'age' in col or 'Age' in col:
            numeric_field = col
            break

    if numeric_field:
        threshold = 20
        # Filtering
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records where {numeric_field} > {threshold}:")
        print(filtered_df.head())

        # Normalization
        filtered_df[f"{numeric_field}_normalized"] = (
            filtered_df[numeric_field] - filtered_df[numeric_field].mean()
        ) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Grouping by key attribute (e.g., hirsutism, use its @id)
        group_field = None
        for col in df.columns:
            if 'hirsutism' in col:
                group_field = col
                break
        if group_field is not None:
            grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
            print(f"Grouped by {group_field}:")
            print(grouped_df.head())
        else:
            print("No suitable grouping field found.")
    else:
        print("No numeric field identified for EDA.")
else:
    print("No record sets loaded for EDA.")

## 5. Visualization
Visualize distributions or relationships, e.g. age at diagnosis vs. presence of hirsutism or infertility.

Below is a visualization using matplotlib and seaborn.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Visualize: Age distribution by hirsutism
if len(record_sets) > 0 and numeric_field and group_field:
    plt.figure(figsize=(8,6))
    sns.boxplot(x=group_field, y=numeric_field, data=df)
    plt.title(f"{numeric_field} distribution by {group_field}")
    plt.xlabel(group_field)
    plt.ylabel(numeric_field)
    plt.show()

    # Scatter plot: Age at diagnosis vs. another binary feature (e.g., infertility)
    infertility_field = None
    for col in df.columns:
        if 'infertility' in col.lower():
            infertility_field = col
            break
    if infertility_field is not None:
        plt.figure(figsize=(8,6))
        sns.scatterplot(x=numeric_field, y=infertility_field, data=df)
        plt.title(f"{numeric_field} vs {infertility_field}")
        plt.xlabel(numeric_field)
        plt.ylabel(infertility_field)
        plt.show()

## 6. Conclusion
This notebook demonstrated how to load and explore the FAIR^2 dataset using mlcroissant, referencing all entities by their `@id`, and applying basic data processing and visualization steps.

Key findings:
- The dataset contains detailed clinical, laboratory, and genetic records from a small cohort of NC-21OHD patients.
- Filtering and normalization can be applied to numeric attributes such as age.
- Grouping by phenotype features (such as hirsutism) helps compare clinical presentation.
- Visualizations illustrate distributions and correlations, but the small N limits inferential analysis.

For further exploration, consider examining hormone levels, imaging results, and genetic variant tables using their respective record sets and fields.